In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load the AnnData object
adata = ad.read_h5ad("./data/cell_cycle/rpe1_kinetics_processed.h5ad")
adata

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM7", "MCM4", "RRM1", "UNG", "GINS2", "MCM6",
    "CDCA7", "DTL", "PRIM1", "UHRF1", "CENPU", "HELLS", "RFC2", "POLR1B", "NASP",
    "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2",
    "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "MRPL36", "E2F8"
]

g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2", "NUF2",
    "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "PIMREG", "SMC4", "CCNB2", "CKAP2L",
    "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
    "CDCA3", "JPT1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5",
    "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5",
    "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]

# --- Step 1: Filter all genes globally ---

# Extract all genes (not just cell cycle subset)
X_all = adata.layers["X_total"].toarray()
V_all = adata.layers["velocity_T"].toarray()

# Expression filter
avg_expr = X_all.mean(axis=0)
expr_threshold = np.quantile(avg_expr, 0.2)
expr_mask = avg_expr > expr_threshold

# Velocity filter
nonzero_velocity_mask = (V_all != 0).any(axis=0)

# Global mask
global_mask = expr_mask & nonzero_velocity_mask

X_filtered = X_all[:, global_mask]
V_filtered = V_all[:, global_mask]
genes_filtered = adata.var_names[global_mask]

phase_numeric = [float(i) for i in list(adata.obs["Cell_cycle_relativePos"])]
phase = list(adata.obs["cell_cycle_phase"])

# --- Step 2: Transform ---
X_log1p = np.log1p(X_filtered)
X_log1p = X_log1p - X_log1p.mean(axis=0, keepdims=True)
V_std = V_filtered / V_filtered.std(axis=0, ddof=0)

# --- Step 3: Subset to cell cycle genes ---
cell_cycle_genes = s_genes + g2m_genes
genes_present = [g for g in cell_cycle_genes if g in genes_filtered]

# Indices of those within the filtered set
gene_indices = [np.where(genes_filtered == g)[0][0] for g in genes_present]

X_cc = X_log1p[:, gene_indices]
V_cc = V_std[:, gene_indices]
genes_cc = np.array(genes_present)

X_cc.shape, V_cc.shape, len(genes_cc)

In [ ]:
import os
import matplotlib.pyplot as plt
import flowmap
from flowmap import *

# ---- Fit embedding ----
emb = VectorFieldEmbedder(
    X_log1p,
    V_std,
    dist_method="phase",
    embed_kwargs={"n_neighbors": 30, "min_dist": 0.3},
    dof=30,
    method="umap"
)
emb.fit_embedding(42)

# ---- Create output directory ----
save_dir = "./figures/cell_cycle"
os.makedirs(save_dir, exist_ok=True)

# ---- Plot ----
fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=phase_numeric,
    grid_density=1.0,
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)

# ---- Save figure (PNG high resolution) ----
save_path = os.path.join(save_dir, "cell_cycle_all_genes_stream.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"Saved to: {save_path}")

In [ ]:
emb = VectorFieldEmbedder(X_cc, V_cc, dist_method="phase",
                          embed_kwargs={"n_neighbors":30,
                                        "min_dist":0.6},
                          dof=30,method="umap")
emb.fit_embedding(1)

fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)

plt.show()

In [ ]:
emb.refine_embedding()

fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from flowmap.utils import compute_velocity_on_grid

def plot_velocity_streamplot(
    X_2d, spline_vf=None, V=None, scatter_color="grey",
    grid_size=50, grid_density=1.0, stream_density=1.0, title=None,
    scatter_size=10, scatter_alpha=0.5, arrowsize=1.5,
    ax=None, figsize=(8, 6), aspect="equal", cmap="tab10",
    vmin=None, vmax=None, show_axes=True, show_colorbar=False,
    streamline_thickness=4.0, show_labels=True, use_cmap=True, cell_type=None,
):

    # --- fit model if not provided ---
    if spline_vf is None:
        if V is None:
            raise ValueError("Either spline_vf or V must be provided.")
        from .TPS import ThinPlateSpline

        # --- subsample if too many points ---
        n_points = X_2d.shape[0]
        max_points = 4000
        if n_points > max_points:
            idx = np.random.choice(n_points, max_points, replace=False)
            X_fit = X_2d[idx]
            V_fit = V[idx]
            print(f"[TPS] Subsampling {max_points}/{n_points} points for fitting …")
        else:
            X_fit = X_2d
            V_fit = V
            print(f"[TPS] Using all {n_points} points for fitting …")

        # --- fit thin-plate spline on 2D subset ---
        spline_vf = ThinPlateSpline(X_fit, n_control_points=100)
        spline_vf.fit(V_fit, dof=15)

    # --- compute filtered grid ---
    Xg, keep, Vg, (xx, yy) = compute_velocity_on_grid(
        X_2d, spline_vf=spline_vf,
        grid_size=grid_size, grid_density=grid_density,
        min_mass=0.01, return_mesh=True
    )
    
    # --- reconstruct full grid ---
    ny, nx = yy.shape[0], xx.shape[1]
    grid_x, grid_y = xx[0, :], yy[:, 0]
    Vx = np.full((ny, nx), np.nan)
    Vy = np.full((ny, nx), np.nan)

    dx = (grid_x[-1] - grid_x[0]) / (nx - 1)
    dy = (grid_y[-1] - grid_y[0]) / (ny - 1)
    j_idx = np.clip(np.rint((Xg[:, 0] - grid_x[0]) / dx).astype(int), 0, nx - 1)
    i_idx = np.clip(np.rint((Xg[:, 1] - grid_y[0]) / dy).astype(int), 0, ny - 1)
    Vx[i_idx, j_idx] = Vg[:, 0]
    Vy[i_idx, j_idx] = Vg[:, 1]

    U = np.ma.masked_invalid(Vx)
    V = np.ma.masked_invalid(Vy)

    # --- plotting ---
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True

    # ---------- robust scatter coloring ----------
    scatter_color = np.array(scatter_color)
    if scatter_color.dtype.kind in {"U", "S", "O"}:  # categorical labels
        unique_vals = np.unique(scatter_color)
        cmap_obj = plt.get_cmap(cmap, len(unique_vals))
        color_map = {val: mcolors.to_hex(cmap_obj(i)) for i, val in enumerate(unique_vals)}
        mapped_colors = np.array([color_map[val] for val in scatter_color])
        ax.scatter(
            X_2d[:, 0], X_2d[:, 1],
            s=scatter_size, alpha=scatter_alpha,
            edgecolors="none", color=mapped_colors
        )
    else:  # numeric or pre-colored array
        if np.issubdtype(scatter_color.dtype, np.number):
            sc = ax.scatter(
                X_2d[:, 0], X_2d[:, 1],
                s=scatter_size, alpha=scatter_alpha,
                c=scatter_color, cmap=cmap, vmin=vmin, vmax=vmax,
                edgecolors="none"
            )
            if show_colorbar:
                plt.colorbar(sc, ax=ax)
        else:
            ax.scatter(
                X_2d[:, 0], X_2d[:, 1],
                s=scatter_size, alpha=scatter_alpha,
                edgecolors="none", color=scatter_color
            )
    # ---------------------------------------------

    # --- streamlines ---
    speed = np.sqrt(Vx**2 + Vy**2)
    smax = np.nanmax(speed) if np.isfinite(speed).any() else 0.0
    linewidth = streamline_thickness * (speed / smax) if smax > 0 else 1.0

    ax.streamplot(
        grid_x, grid_y, U, V,
        linewidth=linewidth,
        density=stream_density,
        color="k",
        arrowsize=arrowsize,
        arrowstyle="-|>",
        maxlength=4,
        integration_direction="both"
    )

    # ----- cell type labels -----
    if cell_type is not None:
        cell_type = np.asarray(cell_type)
        for ct in np.unique(cell_type):
            idx = cell_type == ct
            if idx.sum() == 0:
                continue

            # center of the cluster
            x_center = np.median(X_2d[idx, 0])
            y_center = np.median(X_2d[idx, 1])

            txt = plt.text(
                x_center,
                y_center,
                str(ct),
                ha="center",
                va="center",
                fontsize=18,
                color="black",
                weight="bold",
                zorder=10,
            )

            # white boundary / outline
            txt.set_path_effects([
                pe.Stroke(linewidth=5.5, foreground="white"),
                pe.Normal(),
            ])

    
    ax.set_aspect(aspect)
    if not show_axes:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    else:
        ax.grid(True, linestyle='--', alpha=0.3)
    if title:
        ax.set_title(title)

    if created_fig:
        plt.tight_layout()
        return fig
    else:
        return ax.figure

save_dir = "./figures/cell_cycle"
save_path = os.path.join(save_dir, "cell_cycle_streamplot.png")
fig = plot_velocity_streamplot(
    X_2d=emb.X_emb,
    spline_vf=emb.spline_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.9,
    scatter_size=100,
    scatter_alpha=0.1,
    streamline_thickness=8.0,
    figsize=(4, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    cell_type=phase,
)

fig.savefig(save_path, bbox_inches="tight")
plt.show()

print(f"Saved to: {save_path}")

In [ ]:
# Fit gene-level splines (optional)
emb.fit_gene_level_splines(
    dof_gene=30,
    dof_vf_gene=30,
    X=X_log1p,
    V=V_std
)

# Automatically uses gene-level splines if present
evaluator = flowmap.evaluation.SplineFitEvaluator(emb, mode="gene")

res = evaluator.evaluate()

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import math


# --- make fonts larger globally ---
plt.rcParams.update({
    "font.size": 22,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 18
})

# ------------------------------------------------------------------
# 1) INCLUDE ALL GENES
# ------------------------------------------------------------------
x_all = np.array(res["expr_corr_gene"])
y_all = np.array(res["vel_corr_gene"])
gene_names_all = adata.var_names[global_mask]

mask = np.isfinite(x_all) & np.isfinite(y_all)

x = x_all[mask]
y = y_all[mask]
gene_names = gene_names_all[mask]

# ------------------------------------------------------------------
# 2) DEFINE CURATED CELL-CYCLE GENES
# ------------------------------------------------------------------
genes_cc = np.array(genes_cc)                 # list of 33 gene names
genes_cc_upper = set(g.upper() for g in genes_cc)

categories = np.array([
    "Cell-cycle genes (n=33)" if g.upper() in genes_cc_upper else "Other genes"
    for g in gene_names
])

ordered_categories = [
    "Cell-cycle genes (n=33)",
    "Other genes"
]

categories = pd.Categorical(
    categories,
    categories=ordered_categories,
    ordered=True
)

# Masks
is_cc = categories == "Cell-cycle genes (n=33)"
is_other = categories == "Other genes"

# ------------------------------------------------------------------
# 3) PLOT
# ------------------------------------------------------------------
sns.set_style("whitegrid")
g = sns.JointGrid(x=x, y=y, height=7)

# ----------------------------
# Background: Other genes
# ----------------------------
sns.scatterplot(
    x=x[is_other],
    y=y[is_other],
    color="lightgrey",
    s=60,
    alpha=0.4,
    edgecolor="none",
    ax=g.ax_joint,
    legend=False
)

# ----------------------------
# Foreground: Cell-cycle genes
# ----------------------------
sns.scatterplot(
    x=x[is_cc],
    y=y[is_cc],
    color="#d62728",          # clean red (matches many cell-cycle figs)
    s=180,
    edgecolor="white",
    linewidth=0.6,
    alpha=0.9,
    ax=g.ax_joint,
    label="Cell-cycle \ngenes (n=33)"
)

# Regression line (all genes)
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    color="crimson",
    line_kws={"lw": 3, "alpha": 0.8},
    ax=g.ax_joint
)

# Marginals
sns.histplot(
    x=x,
    bins=30,
    element="step",
    color="grey",
    alpha=0.5,
    ax=g.ax_marg_x
)
sns.histplot(
    y=y,
    bins=30,
    element="step",
    color="grey",
    alpha=0.5,
    ax=g.ax_marg_y
)


# Additional guidelines at 0.3
g.ax_joint.axhline(
    0.3,
    color="black",
    ls=":",
    lw=4,
    alpha=0.8
)

g.ax_joint.axvline(
    0.3,
    color="black",
    ls=":",
    lw=4,
    alpha=0.8
)

# Remove axis labels
g.set_axis_labels("", "")

# ------------------------------------------------------------------
# 4) LEGEND (simple, clean)
# ------------------------------------------------------------------
g.ax_joint.legend(
    loc="lower right",
    bbox_to_anchor=(0.8, 0.02),
    frameon=False,
    ncol=1,
    handletextpad=0.4,
    borderaxespad=0.0
)

# Custom axis ticks
g.ax_joint.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
g.ax_joint.set_yticks([-0.2, 0.0, 0.2, 0.4, 0.6])

# Optional limits for cleaner framing
# g.ax_joint.set_xlim(0, 1.0)
# g.ax_joint.set_ylim(-0.2, 0.6)

plt.tight_layout()

# ---- Create save directory ----
save_dir = "./figures/cell_cycle"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "gene_expression_velocity_correlation.pdf")

# ---- Save figure ----
g.figure.savefig(save_path, bbox_inches="tight")

print(f"Saved to: {save_path}")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

# ============================================================
# 0. Utilities
# ============================================================

def zscore(X):
    """Per-gene z-score (columns). For visualization safety."""
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

def cluster_genes(X):
    """Cluster genes (columns) using Ward linkage."""
    link = linkage(X.T, method="ward")
    return X[:, leaves_list(link)]

# ============================================================
# 1. Cell order & gene masks
# ============================================================

order = np.argsort(phase_numeric)

expr_corr = np.asarray(res["expr_corr_gene"])
vel_corr  = np.asarray(res["vel_corr_gene"])

good_mask = (expr_corr > 0.3) & (vel_corr > 0.3)
bad_mask  = (expr_corr < 0.3) & (vel_corr < 0.3)

# ============================================================
# 2. Extract expression (already normalized upstream,
#    but re-normalized here for visualization robustness)
# ============================================================

X_good = emb.X_gene[order][:, good_mask]
X_bad  = emb.X_gene[order][:, bad_mask]

X_good = zscore(X_good)
X_bad  = zscore(X_bad)

# ============================================================
# 3. Circular smoothing along phase order
# ============================================================

X_good = gaussian_filter1d(X_good, sigma=10, axis=0, mode="wrap")
X_bad  = gaussian_filter1d(X_bad,  sigma=10, axis=0, mode="wrap")

# ============================================================
# 4. Cluster genes independently
# ============================================================

X_good = cluster_genes(X_good)
X_bad  = cluster_genes(X_bad)

# ============================================================
# 5. Circular phase boundary inference
# ============================================================

phases = np.asarray(phase)[order]
n_cells = len(phases)

theta = np.linspace(0, 2*np.pi, n_cells, endpoint=False)

phase_progression = ["S", "G2-M", "M", "M-G1", "G1-S"]

def circular_mean(angles):
    return np.arctan2(
        np.mean(np.sin(angles)),
        np.mean(np.cos(angles))
    ) % (2*np.pi)

# Mean angle per phase
phase_angle = {}
for p in phase_progression:
    mask = phases == p
    phase_angle[p] = circular_mean(theta[mask])

# Order phases by angular position
ordered_phases = sorted(phase_progression, key=lambda p: phase_angle[p])

# Anchor + unwrap
anchor = ordered_phases[0]
anchor_angle = phase_angle[anchor]

def unwrap(a):
    return (a - anchor_angle) % (2*np.pi)

phase_angle_u = {p: unwrap(phase_angle[p]) for p in ordered_phases}

# Progressive right boundaries
boundary_angles = []
current = []
for p in ordered_phases:
    current.append(p)
    right = max(phase_angle_u[q] for q in current)
    boundary_angles.append((right + anchor_angle) % (2*np.pi))

# ============================================================
# 6. Polar heatmap visualization
# ============================================================

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="polar")

theta_edges = np.linspace(0, 2*np.pi, n_cells + 1)

# ----------------------------
# Inner track: bad genes
# ----------------------------
r_bad_inner, r_bad_outer = 0.58, 0.75
r_bad_edges = np.linspace(r_bad_inner, r_bad_outer, X_bad.shape[1] + 1)
T_bad, R_bad = np.meshgrid(theta_edges, r_bad_edges)

ax.pcolormesh(
    T_bad, R_bad, X_bad.T,
    cmap="RdBu_r",
    vmin=-1.5, vmax=1.5,
    shading="flat"
)

# ----------------------------
# Outer track: good genes
# ----------------------------
r_good_inner, r_good_outer = 0.83, 1.0
r_good_edges = np.linspace(r_good_inner, r_good_outer, X_good.shape[1] + 1)
T_good, R_good = np.meshgrid(theta_edges, r_good_edges)

ax.pcolormesh(
    T_good, R_good, X_good.T,
    cmap="RdBu_r",
    vmin=-1.5, vmax=1.5,
    shading="flat"
)

# ----------------------------
# Phase boundaries (piercing)
# ----------------------------
r_circle = r_bad_inner - 0.25

for ang in boundary_angles:
    ax.plot(
        [ang, ang],
        [r_circle, r_good_outer + 0.02],
        linestyle=":",
        color="gray",
        linewidth=2.4,
        alpha=0.8,
        zorder=10
    )

theta_dense = np.linspace(0, 2*np.pi, 512)

ax.plot(
    theta_dense,
    np.full_like(theta_dense, r_circle),
    linestyle=":",
    color="gray",
    linewidth=2.4,
    alpha=0.8,
    zorder=10
)

for r in [
    (r_bad_outer + r_good_inner) / 2,
    r_good_outer + 0.03,
    r_bad_inner - 0.03
]:
    ax.plot(
        theta_dense,
        np.full_like(theta_dense, r),
        color="gray",
        linewidth=3,
        alpha=1.0,
        zorder=10
    )

# ============================================================
# Optional: Phase labels (commented out for manual tweaking)
# ============================================================

# prepend start and append wrap-around end
# bounds = np.array(boundary_angles)
# bounds_ext = np.concatenate([bounds, [bounds[0] + 2*np.pi]])

# # mid-angle for each phase sector
# label_angles = []
# for i in range(len(bounds)):
#     a0 = bounds_ext[i]
#     a1 = bounds_ext[i + 1]
#     label_angles.append((a0 + a1) / 2.0)

# # radial position for text
# r_annot = r_bad_inner - 0.12

# for p, ang in zip(ordered_phases, label_angles):
#     ax.text(
#         ang,
#         r_annot,
#         p,
#         ha="center",
#         va="center",
#         fontsize=12,
#         fontweight="bold",
#         rotation=np.degrees(-ang + np.pi / 2),
#         rotation_mode="anchor",
#         color="black",
#         alpha=0.9,
#         zorder=20
#     )


# ----------------------------
# Polar aesthetics
# ----------------------------
ax.set_theta_direction(-1)
ax.set_theta_zero_location("N")
ax.set_ylim(0, 1.05)
ax.set_xticks([])
ax.set_yticks([])
ax.spines["polar"].set_visible(False)

plt.tight_layout()

# ---- Create save directory ----
save_dir = "./figures/cell_cycle"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "polar_phase_heatmap.png")

# ---- Save high-resolution PNG ----
fig.savefig(
    save_path,
    dpi=600,                 # High resolution
    bbox_inches="tight",
    facecolor="white"        # Ensures solid background
)

print(f"Saved to: {save_path}")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

# ============================================================
# 0. Utilities
# ============================================================

def zscore(X):
    """Per-gene z-score (columns). For visualization safety."""
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

def cluster_genes(X):
    """Cluster genes (columns) using Ward linkage."""
    link = linkage(X.T, method="ward")
    return X[:, leaves_list(link)]

# ============================================================
# 1. Cell order & gene masks
# ============================================================

order = np.argsort(phase_numeric)

expr_corr = np.asarray(res["expr_corr_gene"])
vel_corr  = np.asarray(res["vel_corr_gene"])


thr_expr = 0.3
thr_vel  = 0.3

mask_hh = (expr_corr > thr_expr) & (vel_corr > thr_vel)
mask_hl = (expr_corr > thr_expr) & (vel_corr <= thr_vel)
mask_lh = (expr_corr <= thr_expr) & (vel_corr > thr_vel)
mask_ll = (expr_corr <= thr_expr) & (vel_corr <= thr_vel)

# ============================================================
# 2. Extract & normalize
# ============================================================

tracks = {
    "HH": emb.X_gene[order][:, mask_hh],
    "HL": emb.X_gene[order][:, mask_hl],
    "LH": emb.X_gene[order][:, mask_lh],
    "LL": emb.X_gene[order][:, mask_ll],
}

for k in tracks:
    tracks[k] = zscore(tracks[k])
    tracks[k] = gaussian_filter1d(
        tracks[k],
        sigma=10,
        axis=0,
        mode="wrap"
    )
    tracks[k] = cluster_genes(tracks[k])

# ============================================================
# 3. Circular smoothing along phase order
# ============================================================

X_good = gaussian_filter1d(X_good, sigma=10, axis=0, mode="wrap")
X_bad  = gaussian_filter1d(X_bad,  sigma=10, axis=0, mode="wrap")

# ============================================================
# 4. Cluster genes independently
# ============================================================

X_good = cluster_genes(X_good)
X_bad  = cluster_genes(X_bad)

# ============================================================
# 5. Circular phase boundary inference
# ============================================================

phases = np.asarray(phase)[order]
n_cells = len(phases)

theta = np.linspace(0, 2*np.pi, n_cells, endpoint=False)

phase_progression = ["S", "G2-M", "M", "M-G1", "G1-S"]

def circular_mean(angles):
    return np.arctan2(
        np.mean(np.sin(angles)),
        np.mean(np.cos(angles))
    ) % (2*np.pi)

# Mean angle per phase
phase_angle = {}
for p in phase_progression:
    mask = phases == p
    phase_angle[p] = circular_mean(theta[mask])

# Order phases by angular position
ordered_phases = sorted(phase_progression, key=lambda p: phase_angle[p])

# Anchor + unwrap
anchor = ordered_phases[0]
anchor_angle = phase_angle[anchor]

def unwrap(a):
    return (a - anchor_angle) % (2*np.pi)

phase_angle_u = {p: unwrap(phase_angle[p]) for p in ordered_phases}

# Progressive right boundaries
boundary_angles = []
current = []
for p in ordered_phases:
    current.append(p)
    right = max(phase_angle_u[q] for q in current)
    boundary_angles.append((right + anchor_angle) % (2*np.pi))

# ============================================================
# 6. Polar heatmap visualization
# ============================================================

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection="polar")

theta_edges = np.linspace(0, 2*np.pi, n_cells + 1)

# ------------------------------------------------
# Ring definitions (inner -> outer)
# ------------------------------------------------

ring_defs = [
    ("LL", 0.38, 0.52),
    ("LH", 0.56, 0.70),
    ("HL", 0.74, 0.88),
    ("HH", 0.92, 1.06),
]

for label, r0, r1 in ring_defs:

    X = tracks[label]

    r_edges = np.linspace(
        r0,
        r1,
        X.shape[1] + 1
    )

    T, R = np.meshgrid(theta_edges, r_edges)

    ax.pcolormesh(
        T,
        R,
        X.T,
        cmap="RdBu_r",
        vmin=-1.5,
        vmax=1.5,
        shading="flat"
    )

# ------------------------------------------------
# Ring separators
# ------------------------------------------------

theta_dense = np.linspace(0, 2*np.pi, 512)

for _, r0, r1 in ring_defs:

    ax.plot(
        theta_dense,
        np.full_like(theta_dense, r0),
        color="gray",
        linewidth=2.5,
        alpha=1.0,
        zorder=10
    )

ax.plot(
    theta_dense,
    np.full_like(theta_dense, ring_defs[-1][2]),
    color="gray",
    linewidth=2.5,
    alpha=1.0,
    zorder=10
)

# ============================================================
# Optional: Phase labels (commented out for manual tweaking)
# ============================================================

# prepend start and append wrap-around end
# bounds = np.array(boundary_angles)
# bounds_ext = np.concatenate([bounds, [bounds[0] + 2*np.pi]])

# # mid-angle for each phase sector
# label_angles = []
# for i in range(len(bounds)):
#     a0 = bounds_ext[i]
#     a1 = bounds_ext[i + 1]
#     label_angles.append((a0 + a1) / 2.0)

# # radial position for text
# r_annot = r_bad_inner - 0.12

# for p, ang in zip(ordered_phases, label_angles):
#     ax.text(
#         ang,
#         r_annot,
#         p,
#         ha="center",
#         va="center",
#         fontsize=12,
#         fontweight="bold",
#         rotation=np.degrees(-ang + np.pi / 2),
#         rotation_mode="anchor",
#         color="black",
#         alpha=0.9,
#         zorder=20
#     )


# ----------------------------
# Polar aesthetics
# ----------------------------
ax.set_theta_direction(-1)
ax.set_theta_zero_location("N")
ax.set_ylim(0, 1.05)
ax.set_xticks([])
ax.set_yticks([])
ax.spines["polar"].set_visible(False)

plt.tight_layout()

# ---- Create save directory ----
save_dir = "./figures/cell_cycle"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "polar_phase_heatmap_4regions.png")

# ---- Save high-resolution PNG ----
fig.savefig(
    save_path,
    dpi=600,                 # High resolution
    bbox_inches="tight",
    facecolor="white"        # Ensures solid background
)

print(f"Saved to: {save_path}")

plt.show()

In [ ]:
from sklearn.decomposition import TruncatedSVD
from flowmap.core.spline import Spline
import numpy as np

# --------------------------------------------------
# 1. PCA projection
# --------------------------------------------------
n_components = 6

svd = TruncatedSVD(n_components=n_components, random_state=0)
X_pc = svd.fit_transform(emb.X)
V_pc = emb.V @ svd.components_.T

# --------------------------------------------------
# 2. Geometry spline (embedding → PC space)
# --------------------------------------------------
print("[Spline] Fitting spline_pc (embedding → PC space) …")

spline_pc = Spline(
    emb.X_emb,
    n_control_points=4000,
)

spline_pc.fit(
    X_pc,
    dof=30,
)

# --------------------------------------------------
# 3. Recover intrinsic velocity
# --------------------------------------------------
print("[Spline] Mapping vector field via spline_pc …")

J = spline_pc.compute_jacobians(emb.X_emb)

v_emb = np.stack([
    np.linalg.lstsq(J[i], V_pc[i], rcond=None)[0]
    for i in range(len(J))
])

# --------------------------------------------------
# 4. Velocity spline (embedding → intrinsic velocity)
# --------------------------------------------------
print("[Spline] Fitting spline_vf_pc (embedding → v_emb) …")

spline_vf_pc = Spline(
    emb.X_emb,
    n_control_points=4000,
)

spline_vf_pc.fit(
    v_emb,
    dof=30,
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ------------------------------------------------------------
# 0) Output directory
# ------------------------------------------------------------
out_dir = "./figures/cell_cycle"
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1) Use first 3 PCs
# ------------------------------------------------------------
selected_PC = np.array([0, 1, 2])

raw_expr  = np.asarray(X_pc[:, selected_PC])
pred_expr = np.asarray(
    spline_pc.predict(emb.X_emb)[:, selected_PC]
)

# ------------------------------------------------------------
# 2) Cell cycle colors
# ------------------------------------------------------------
phase = np.asarray(phase_numeric)

norm = mcolors.Normalize(vmin=phase.min(), vmax=phase.max())
cmap = plt.get_cmap("viridis")
cell_colors = cmap(norm(phase))

# ------------------------------------------------------------
# 3) PCA metric scaling
# ------------------------------------------------------------
singular_vals = svd.singular_values_
pc_scales = singular_vals[selected_PC]

# ------------------------------------------------------------
# 4) Consistent axis limits
# ------------------------------------------------------------
xmin = min(raw_expr[:,0].min(), pred_expr[:,0].min())
xmax = max(raw_expr[:,0].max(), pred_expr[:,0].max())

ymin = min(raw_expr[:,1].min(), pred_expr[:,1].min())
ymax = max(raw_expr[:,1].max(), pred_expr[:,1].max())

zmin = min(raw_expr[:,2].min(), pred_expr[:,2].min())
zmax = max(raw_expr[:,2].max(), pred_expr[:,2].max())

# ------------------------------------------------------------
# 5) Polished 3D scatter (saving version)
# ------------------------------------------------------------
def scatter3d_save(data, colors, filename, title=""):

    fig = plt.figure(figsize=(6, 5), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = data.T

    ax.scatter(
        x, y, z,
        c=colors,
        s=12,
        alpha=0.9,
        edgecolors="none"
    )

    ax.set_title(title, fontsize=12, pad=10)
    ax.view_init(elev=25, azim=-120)

    # PCA-metric-correct geometry
    ax.set_box_aspect(pc_scales)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_zlim(zmin, zmax)

    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    plt.tight_layout()

    fig.savefig(
        os.path.join(out_dir, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# ------------------------------------------------------------
# 6) Save both plots
# ------------------------------------------------------------
scatter3d_save(
    raw_expr,
    cell_colors,
    "cell_cycle_raw_PC1-3.pdf"
)

scatter3d_save(
    pred_expr,
    cell_colors,
    "cell_cycle_spline_PC1-3.pdf"
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from flowmap.evaluation.spline_fit_evaluator import SplineFitEvaluator

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
outdir = "./figures/cell_cycle"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# DOF sweep
# ------------------------------------------------------------
dofs = [30, 40, 50, 75, 100, 150]

expr_r2_list = []
vel_r2_list  = []

for dof in dofs:
    print(f"\n[DOF = {dof}] Fitting gene-level splines...")

    # Fit gene-level splines
    emb.fit_gene_level_splines(
        dof_gene=dof,
        dof_vf_gene=dof,
        X=emb.X,
        V=emb.V
    )

    # Evaluate reconstruction
    evaluator = SplineFitEvaluator(emb, mode="gene")
    metrics = evaluator.evaluate()

    expr_r2 = metrics["expr_r2"]
    vel_r2  = metrics["vel_r2"]

    expr_r2_list.append(expr_r2)
    vel_r2_list.append(vel_r2)

    print(f"  Expression R² = {expr_r2:.4f}")
    print(f"  Velocity   R² = {vel_r2:.4f}")

expr_r2_arr = np.array(expr_r2_list)
vel_r2_arr  = np.array(vel_r2_list)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(7, 6))

plt.plot(
    dofs,
    expr_r2_arr,
    marker="o",
    lw=2.5,
    ms=8,
    label="Expression",
)

plt.plot(
    dofs,
    vel_r2_arr,
    marker="s",
    lw=2.5,
    ms=8,
    label="Velocity",
)

plt.xlabel("Spline degrees of freedom")
plt.ylabel("Variance explained ($R^2$)")
plt.grid(alpha=0.3)
plt.legend(frameon=False)

plt.tight_layout()

# Save
save_path = os.path.join(outdir, "dof_sweep.pdf")
plt.savefig(save_path)
plt.close()

print(f"\nSaved figure to: {save_path}")

In [ ]:
from flowmap import *
from flowmap.geometry import FixedPointAnalyzer
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Identify fixed points
# ------------------------------------------------------------
analyzer = FixedPointAnalyzer(emb)

fps = analyzer.identify_fixed_points(
    grid_size=50,
    grid_density=1.0,
    global_speed_quantile=0.05,
)

print(f"Found {len(fps)} fixed points")

# ------------------------------------------------------------
# 2. Plot embedding + highlight fixed points
# ------------------------------------------------------------
plt.figure(figsize=(7, 6))

# background cells
plt.scatter(
    emb.X_emb[:, 0],
    emb.X_emb[:, 1],
    s=6,
    alpha=0.3,
    color="gray"
)

# highlight fixed points
for fp in fps:
    pos = fp["position"]
    plt.scatter(
        pos[0],
        pos[1],
        s=120,
        c="red",
        edgecolor="black",
        linewidth=1.5,
        zorder=5,
    )

plt.title("Fixed Points")
plt.xlabel("Embedding 1")
plt.ylabel("Embedding 2")
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
save_dir = "./figures/cell_cycle"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "cell_cycle_circular_alignment.pdf")

# ============================================================
# JS circular–circular correlation
# ============================================================

def circular_correlation_js(alpha, beta):
    """
    Jammalamadaka–Sengupta circular–circular correlation coefficient.
    alpha, beta: angles in radians.
    """
    alpha_bar = np.angle(np.mean(np.exp(1j * alpha)))
    beta_bar  = np.angle(np.mean(np.exp(1j * beta)))

    num = np.sum(
        np.sin(alpha - alpha_bar) * np.sin(beta - beta_bar)
    )
    den = np.sqrt(
        np.sum(np.sin(alpha - alpha_bar) ** 2) *
        np.sum(np.sin(beta - beta_bar) ** 2)
    )
    return num / den


# ============================================================
# Prepare angles
# ============================================================

X = emb.X_emb
gt = np.array(phase_numeric)          # in [0,1]
center = fps[1]["position"]                   # chosen reference center

# embedding angle
X_shifted = X - center
theta_emb = np.arctan2(X_shifted[:, 1], X_shifted[:, 0])
theta_emb = (theta_emb + 2*np.pi) % (2*np.pi)

# ground-truth angle
theta_gt = (gt % 1.0) * 2*np.pi


# ============================================================
# JS circular correlation
# ============================================================

circ_corr = circular_correlation_js(theta_emb, theta_gt)
print(f"JS circular correlation = {circ_corr:.3f}")


# ============================================================
# Circular difference after global alignment
# ============================================================

# raw circular difference
delta_raw = np.angle(np.exp(1j * (theta_emb - theta_gt)))

# global phase offset
phase_offset = np.angle(np.mean(np.exp(1j * delta_raw)))

# aligned difference
delta_aligned = np.angle(np.exp(1j * (delta_raw - phase_offset)))


# ============================================================
# Plot histogram (publication-ready)
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

ax.hist(
    delta_aligned,
    bins=40,
    density=True,
    color="steelblue",
    alpha=0.85
)

ax.axvline(0, color="black", lw=2, ls="--")

ax.set_xlabel(r"Aligned circular difference $\Delta\theta$ (rad)")
ax.set_ylabel("Density")
ax.set_title(f"|JS circular corr| = {abs(circ_corr):.2f}")

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved to: {save_path}")